In [1]:
import os
CODE_DIR = "/kaggle/working"
DATASET  = "MinhDuk/Chunking_571_Skynet_combined"
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["HF_LEROBOT_HOME"] = "/tmp/lerobot"
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
!pip install -q "lerobot[smolvla] @ git+https://github.com/huggingface/lerobot.git" av
!nvidia-smi -L

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
GPU 0: Tesla T4 (UUID: GPU-f909ed80-c887-0dd5-42a0-ebf65a1792e9)
GPU 1: Tesla T4 (UUID: GPU-f461f36f-8080-a83a-2456-de1c7aa1cc4f)


In [3]:
%%writefile /kaggle/working/fox_patch.py
#!/usr/bin/env python
"""
Patch LeRobot's SmolVLA in place so that the action expert's *self-attention*
layers use FoX (Forgetting Attention, Lin et al., ICLR 2025).

What is changed (everything else is untouched):
  * configuration_smolvla.py   : new flags use_fox / fox_bias_init / fox_identity_init / fox_alpha_lr
  * smolvlm_with_expert.py     : ForgetGate module, gate bias added to the action->action block of
                                 the self-attention layers (even layers), VLM loaded in fp32
  * modeling_smolvla.py        : flags passed through, model cast to fp32, optimizer param groups
                                 (gate alpha gets its own LR, no weight decay on gates)

Cross-attention layers (odd layers), KV cache, masks, RoPE, flow-matching loss: unchanged.

Usage:
    python fox_patch.py                       # patches the installed lerobot
    python fox_patch.py --dir path/to/smolvla # patches a given folder
    python fox_patch.py --revert              # restores the .orig backups
"""
import argparse
import importlib.util
import pathlib
import shutil
import sys


def find_dir() -> pathlib.Path:
    spec = importlib.util.find_spec("lerobot")
    if spec is None or not spec.submodule_search_locations:
        sys.exit("lerobot is not installed. pip install 'lerobot[smolvla]' first.")
    return pathlib.Path(list(spec.submodule_search_locations)[0]) / "policies" / "smolvla"


def sub(text: str, old: str, new: str, name: str) -> str:
    n = text.count(old)
    if n != 1:
        sys.exit(f"[patch failed] anchor '{name}' found {n}x (expected 1). "
                 f"Your lerobot version differs from the one this patch was written for.")
    return text.replace(old, new)


# --------------------------------------------------------------------------------------
# configuration_smolvla.py
# --------------------------------------------------------------------------------------
def patch_config(t: str) -> str:
    return sub(
        t,
        '    compile_mode: str = "max-autotune"  # Torch compile mode\n',
        '    compile_mode: str = "max-autotune"  # Torch compile mode\n'
        "\n"
        "    # FoX (Forgetting Attention) in the action expert's self-attention layers\n"
        "    use_fox: bool = False\n"
        "    fox_bias_init: float = 4.0  # sigmoid(4) = 0.982\n"
        "    fox_identity_init: bool = True  # alpha = 0 -> exactly the baseline at step 0\n"
        "    fox_alpha_lr: float = 2e-3  # Adam moves ~LR per step, so alpha needs a larger LR than 1e-4\n",
        "config.fields",
    )


# --------------------------------------------------------------------------------------
# smolvlm_with_expert.py
# --------------------------------------------------------------------------------------
FORGET_GATE = '''import torch.nn.functional as F  # noqa: E402


class ForgetGate(nn.Module):
    """FoX forget gate. Returns an additive attention bias alpha_h * D, D_ij = c_i - c_j,
    c = cumsum(log f), log f = logsigmoid(W h + b), computed in fp32, one gate per query head.
    alpha_h is clamped >= 0 and initialised to 0 (identity at init) when identity_init=True."""

    def __init__(self, hidden, heads, bias_init=4.0, identity_init=True):
        super().__init__()
        self.proj = nn.Linear(hidden, heads)
        nn.init.normal_(self.proj.weight, std=0.02)
        nn.init.constant_(self.proj.bias, bias_init)
        self.alpha = nn.Parameter(torch.zeros(heads) if identity_init else torch.ones(heads))

    def forward(self, x):  # x: [B, Ls, E] normed expert hidden states
        log_f = F.logsigmoid(self.proj(x.float()))  # [B, Ls, H]
        c = torch.cumsum(log_f, dim=1).transpose(1, 2)  # [B, H, Ls]
        d = c[:, :, :, None] - c[:, :, None, :]  # [B, H, Ls, Ls], zero diagonal
        return self.alpha.clamp(min=0)[None, :, None, None] * d


class SmolVLMWithExpertModel(nn.Module):'''

GATE_BUILD = '''        # Remove unused embed_tokens
        self.lm_expert.embed_tokens = None

        # FoX gates: one per self-attention layer of the action expert
        self.fox_gates = nn.ModuleDict()
        if use_fox:
            assert self.num_expert_layers == self.num_vlm_layers, (
                "FoX gates assume num_expert_layers == num_vlm_layers (default config)"
            )
            for _i in range(self.num_vlm_layers):
                _is_self = ("cross" not in attention_mode) or (
                    self_attn_every_n_layers > 0 and _i % self_attn_every_n_layers == 0
                )
                if _is_self:
                    self.fox_gates[str(_i)] = ForgetGate(
                        lm_expert_config.hidden_size,
                        config.text_config.num_attention_heads,
                        fox_bias_init,
                        fox_identity_init,
                    )
'''


def patch_vlm(t: str) -> str:
    t = sub(t, "class SmolVLMWithExpertModel(nn.Module):", FORGET_GATE, "class")
    t = sub(t, 'torch_dtype="bfloat16",', 'torch_dtype="float32",', "fp32 loader")
    t = sub(
        t,
        '        device: str = "auto",\n    ):\n        super().__init__()\n        require_package("transformers", extra="smolvla")',
        '        device: str = "auto",\n'
        "        use_fox: bool = False,\n"
        "        fox_bias_init: float = 4.0,\n"
        "        fox_identity_init: bool = True,\n"
        '    ):\n        super().__init__()\n        require_package("transformers", extra="smolvla")',
        "init signature",
    )
    t = sub(
        t,
        "        # Remove unused embed_tokens\n        self.lm_expert.embed_tokens = None\n",
        GATE_BUILD,
        "gate build",
    )
    # forward_attn_layer: init gate_bias
    t = sub(
        t,
        "        query_states = []\n        key_states = []\n        value_states = []\n",
        "        gate_bias = None\n        query_states = []\n        key_states = []\n        value_states = []\n",
        "gate_bias init",
    )
    # forward_attn_layer: compute the gate from the normed expert hidden state
    t = sub(
        t,
        "            hidden_states = layer.input_layernorm(hidden_states)\n",
        "            hidden_states = layer.input_layernorm(hidden_states)\n"
        "            if i == 1 and str(layer_idx) in self.fox_gates:  # expert (action) tokens only\n"
        "                gate_bias = self.fox_gates[str(layer_idx)](hidden_states)\n",
        "gate compute",
    )
    # forward_attn_layer: pad the [B,H,Ls,Ls] bias to [B,H,Lq,Lk] (zeros on prefix rows/cols) and pass it on
    t = sub(
        t,
        "        att_output = attention_interface(\n"
        "            attention_mask_, batch_size, head_dim, query_states, key_states, value_states\n"
        "        )\n",
        "        att_bias = None\n"
        "        if gate_bias is not None:\n"
        "            _ls, _lq, _lk = gate_bias.shape[-1], query_states.shape[1], key_states.shape[1]\n"
        "            att_bias = F.pad(gate_bias, (_lk - _ls, 0, _lq - _ls, 0))\n"
        "        att_output = attention_interface(\n"
        "            attention_mask_, batch_size, head_dim, query_states, key_states, value_states, att_bias=att_bias\n"
        "        )\n",
        "attention call",
    )
    # eager attention: accept and add the bias
    t = sub(
        t,
        "    def eager_attention_forward(\n"
        "        self, attention_mask, batch_size, head_dim, query_states, key_states, value_states\n"
        "    ):\n",
        "    def eager_attention_forward(\n"
        "        self, attention_mask, batch_size, head_dim, query_states, key_states, value_states, att_bias=None\n"
        "    ):\n",
        "eager signature",
    )
    t = sub(
        t,
        "        att_weights = att_weights.to(dtype=torch.float32)\n        big_neg",
        "        att_weights = att_weights.to(dtype=torch.float32)\n"
        "        if att_bias is not None:\n"
        "            att_weights = att_weights + att_bias\n"
        "        big_neg",
        "eager bias add",
    )
    return t


# --------------------------------------------------------------------------------------
# modeling_smolvla.py
# --------------------------------------------------------------------------------------
OPTIM = '''    def get_optim_params(self):
        if not getattr(self.config, "use_fox", False):
            return self.parameters()
        alpha, gate_w, rest = [], [], []
        for n, p in self.named_parameters():
            if not p.requires_grad:
                continue
            if "fox_gates" in n:
                (alpha if n.endswith(".alpha") else gate_w).append(p)
            else:
                rest.append(p)
        groups = [{"params": rest}]
        if gate_w:  # base LR, no weight decay
            groups.append({"params": gate_w, "weight_decay": 0.0})
        if alpha:  # larger LR, no weight decay
            groups.append({"params": alpha, "lr": self.config.fox_alpha_lr, "weight_decay": 0.0})
        return groups'''


def patch_modeling(t: str) -> str:
    t = sub(
        t,
        "    def get_optim_params(self) -> dict:\n        return self.parameters()",
        OPTIM,
        "optim params",
    )
    t = sub(
        t,
        '            device=self.config.device if self.config.device is not None else "auto",\n        )\n',
        '            device=self.config.device if self.config.device is not None else "auto",\n'
        "            use_fox=self.config.use_fox,\n"
        "            fox_bias_init=self.config.fox_bias_init,\n"
        "            fox_identity_init=self.config.fox_identity_init,\n"
        "        )\n",
        "pass flags",
    )
    t = sub(
        t,
        "        self.prefix_length = self.config.prefix_length\n        self.rtc_processor = rtc_processor\n",
        "        self.prefix_length = self.config.prefix_length\n        self.rtc_processor = rtc_processor\n"
        "        self.float()  # fp32 everywhere (T4 has no native bf16)\n",
        "fp32 cast",
    )
    return t


PATCHES = {
    "configuration_smolvla.py": patch_config,
    "smolvlm_with_expert.py": patch_vlm,
    "modeling_smolvla.py": patch_modeling,
}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dir", default=None)
    ap.add_argument("--revert", action="store_true")
    args = ap.parse_args()
    d = pathlib.Path(args.dir) if args.dir else find_dir()
    print("SmolVLA dir:", d)

    if args.revert:
        for name in PATCHES:
            bak = d / (name + ".orig")
            if bak.exists():
                shutil.copy(bak, d / name)
                print("restored", name)
        return

    if "use_fox" in (d / "configuration_smolvla.py").read_text():
        print("Already patched, nothing to do.")
        return

    new = {}
    for name, fn in PATCHES.items():  # patch in memory first, write only if ALL succeed
        new[name] = fn((d / name).read_text())
    for name, text in new.items():
        shutil.copy(d / name, d / (name + ".orig"))
        (d / name).write_text(text)
        print("patched", name)


if __name__ == "__main__":
    main()

Writing /kaggle/working/fox_patch.py


In [4]:
%%writefile /kaggle/working/fox_sanity_test.py
#!/usr/bin/env python
"""
Safety tests for the FoX patch. Run BEFORE training (after fox_patch.py).

    python fox_sanity_test.py            # math tests only (CPU, seconds)
    python fox_sanity_test.py --model    # + model-level tests on lerobot/smolvla_base (needs a GPU)

Math tests      : paper formula, zero diagonal, causality.
Model-level     : (1) alpha=0 reproduces the baseline exactly (train loss and sampled actions)
                  (2) full uncached forward == prefill + cached denoise_step with alpha=1 / random gates
                  (3) alpha receives a gradient at init, gates are finite (fp32), no NaN
"""
import argparse
import math

import torch

from lerobot.policies.smolvla.smolvlm_with_expert import ForgetGate

MODEL_ID = "lerobot/smolvla_base"


# ------------------------------------------------------------------------------ math
def test_math():
    torch.manual_seed(0)
    B, H, L, D, E = 2, 3, 12, 16, 24
    gate = ForgetGate(E, H, bias_init=0.0, identity_init=False)  # alpha = 1
    nn_init = torch.nn.init.normal_
    nn_init(gate.proj.weight, std=0.5)
    x = torch.randn(B, L, E)
    q, k, v = (torch.randn(B, H, L, D) for _ in range(3))

    bias = gate(x)  # [B,H,L,L]
    assert torch.allclose(torch.diagonal(bias, dim1=-2, dim2=-1), torch.zeros(B, H, L)), "diagonal must be 0"

    causal = torch.tril(torch.ones(L, L, dtype=torch.bool))
    s = q @ k.transpose(-1, -2) / math.sqrt(D) + bias
    p = torch.softmax(s.masked_fill(~causal, torch.finfo(s.dtype).min), dim=-1)
    out_bias = p @ v

    # explicit paper formula: o_i = sum_j F_ij exp(q_i.k_j) v_j / sum_j F_ij exp(q_i.k_j), F_ij = prod_{l=j+1..i} f_l
    f = torch.sigmoid(gate.proj(x)).transpose(1, 2)  # [B,H,L]
    out_ref = torch.zeros_like(out_bias)
    for i in range(L):
        w = []
        for j in range(i + 1):
            Fij = f[:, :, j + 1 : i + 1].prod(-1) if j < i else torch.ones(B, H)
            w.append(Fij * torch.exp((q[:, :, i] * k[:, :, j]).sum(-1) / math.sqrt(D)))
        w = torch.stack(w, -1)  # [B,H,i+1]
        out_ref[:, :, i] = (w[..., None] * v[:, :, : i + 1]).sum(-2) / w.sum(-1, keepdim=True)
    assert torch.allclose(out_bias, out_ref, atol=1e-5), (out_bias - out_ref).abs().max()

    # causality: changing input token t+1.. must not change bias rows <= t (lower triangle)
    t = 5
    x2 = x.clone()
    x2[:, t + 1 :] += torch.randn_like(x2[:, t + 1 :])
    b2 = gate(x2)
    assert torch.allclose(bias[:, :, : t + 1, : t + 1], b2[:, :, : t + 1, : t + 1], atol=1e-6)

    # identity init
    g0 = ForgetGate(E, H, identity_init=True)
    assert gate_is_zero(g0(x))
    print("[ok] math tests: paper formula, zero diagonal, causality, identity init")


def gate_is_zero(b):
    return bool((b == 0).all())


# ------------------------------------------------------------------------------ model-level
def fake_batch(policy, B=2, device="cuda"):
    from lerobot.utils.constants import ACTION, OBS_LANGUAGE_ATTENTION_MASK, OBS_LANGUAGE_TOKENS, OBS_STATE

    cfg = policy.config
    batch = {}
    for key, ft in cfg.image_features.items():
        c, h, w = ft.shape
        batch[key] = torch.rand(B, c, h, w, device=device)
    batch[OBS_STATE] = torch.randn(B, cfg.robot_state_feature.shape[0], device=device)
    batch[ACTION] = torch.randn(B, cfg.chunk_size, cfg.action_feature.shape[0], device=device)
    batch[OBS_LANGUAGE_TOKENS] = torch.randint(0, 1000, (B, 16), device=device)
    batch[OBS_LANGUAGE_ATTENTION_MASK] = torch.ones(B, 16, dtype=torch.bool, device=device)
    return batch


def load_policy(use_fox, device="cuda"):
    from lerobot.configs import PreTrainedConfig
    from lerobot.policies.smolvla import SmolVLAPolicy

    cfg = PreTrainedConfig.from_pretrained(MODEL_ID)
    cfg.device = device
    cfg.use_fox = use_fox
    cfg.fox_identity_init = True
    policy = SmolVLAPolicy.from_pretrained(MODEL_ID, config=cfg).to(device)
    return policy


def unpack(policy, batch):
    from lerobot.utils.constants import OBS_LANGUAGE_ATTENTION_MASK, OBS_LANGUAGE_TOKENS

    images, img_masks = policy.prepare_images(batch)
    state = policy.prepare_state(batch)
    actions = policy.prepare_action(batch)
    return images, img_masks, batch[OBS_LANGUAGE_TOKENS], batch[OBS_LANGUAGE_ATTENTION_MASK], state, actions


@torch.no_grad()
def uncached_vt(vf, images, img_masks, lang, lm, state, x_t, time):
    from lerobot.policies.common.vla_utils import make_att_2d_masks

    pe, ppm, pam = vf.embed_prefix(images, img_masks, lang, lm, state=state)
    se, spm, sam = vf.embed_suffix(x_t, time)
    pad = torch.cat([ppm, spm], 1)
    att = torch.cat([pam, sam], 1)
    (_, suf), _ = vf.vlm_with_expert.forward(
        attention_mask=make_att_2d_masks(pad, att),
        position_ids=torch.cumsum(pad, 1) - 1,
        past_key_values=None,
        inputs_embeds=[pe, se],
        use_cache=False,
    )
    return vf.action_out_proj(suf[:, -vf.config.chunk_size :].float())


@torch.no_grad()
def cached_vt(vf, images, img_masks, lang, lm, state, x_t, time):
    from lerobot.policies.common.vla_utils import make_att_2d_masks

    pe, ppm, pam = vf.embed_prefix(images, img_masks, lang, lm, state=state)
    _, pkv = vf.vlm_with_expert.forward(
        attention_mask=make_att_2d_masks(ppm, pam),
        position_ids=torch.cumsum(ppm, 1) - 1,
        past_key_values=None,
        inputs_embeds=[pe, None],
        use_cache=True,
    )
    return vf.denoise_step(prefix_pad_masks=ppm, past_key_values=pkv, x_t=x_t, timestep=time)


def test_model():
    from lerobot.policies.common.flow_matching import FlowConvention, make_flow_matching_inputs

    dev = "cuda"
    base = load_policy(False, dev).eval()
    fox = load_policy(True, dev).eval()
    # make both models identical outside the gates (fox was loaded from the same checkpoint)
    miss = [k for k in fox.state_dict() if k not in base.state_dict()]
    assert miss and all("fox_gates" in k for k in miss), f"unexpected extra keys: {miss[:5]}"
    print(f"[info] {len(miss)} new tensors, all fox_gates.*")

    torch.manual_seed(0)
    batch = fake_batch(base, device=dev)
    B = batch[next(iter(base.config.image_features))].shape[0]
    noise = torch.randn(B, base.config.chunk_size, base.config.max_action_dim, device=dev)
    time = torch.rand(B, device=dev) * 0.9 + 0.05

    ia = unpack(base, dict(batch))
    ib = unpack(fox, dict(batch))

    # (1) alpha = 0  ->  baseline, exactly
    with torch.no_grad():
        la = base.model.forward(*ia, noise, time)
        lb = fox.model.forward(*ib, noise, time)
        assert torch.allclose(la, lb, atol=1e-6), (la - lb).abs().max()
        sa = base.model.sample_actions(*ia[:5], noise=noise)
        sb = fox.model.sample_actions(*ib[:5], noise=noise)
        assert torch.allclose(sa, sb, atol=1e-6), (sa - sb).abs().max()
    print("[ok] (1) alpha=0 reproduces the baseline (loss + sampled actions)")

    # (3) alpha gets gradient at init, proj gets zero grad (alpha=0), everything finite
    fox.train()
    fox.zero_grad()
    loss = fox.model.forward(*ib, noise, time).mean()
    loss.backward()
    a_grads = [g.alpha.grad for g in fox.model.vlm_with_expert.fox_gates.values()]
    assert all(g is not None and torch.isfinite(g).all() for g in a_grads)
    assert any(g.abs().sum() > 0 for g in a_grads), "alpha receives no gradient -> it could never leave 0"
    assert torch.isfinite(loss)
    print(f"[ok] (3) alpha grad non-zero at init, loss finite ({loss.item():.4f})")
    fox.eval()
    fox.zero_grad(set_to_none=True)

    # (2) alpha = 1, random gates: uncached training path == prefill + cached denoise path
    for g in fox.model.vlm_with_expert.fox_gates.values():
        torch.nn.init.normal_(g.proj.weight, std=0.5)
        torch.nn.init.normal_(g.proj.bias, std=1.0)
        g.alpha.data.fill_(1.0)
    x_t, _, _ = make_flow_matching_inputs(ib[5], noise, time, convention=FlowConvention.NOISE_AT_ONE)
    v_unc = uncached_vt(fox.model, *ib[:5], x_t, time)
    v_cac = cached_vt(fox.model, *ib[:5], x_t, time)
    err = (v_unc - v_cac).abs().max().item()
    assert err < 1e-3, f"train path and cached path disagree (max abs err {err:.3e}) -> bias padding is wrong"
    # and the gate must actually change the output
    for g in fox.model.vlm_with_expert.fox_gates.values():
        g.alpha.data.zero_()
    v0 = uncached_vt(fox.model, *ib[:5], x_t, time)
    assert (v0 - v_unc).abs().max() > 1e-4, "gate has no effect"
    print(f"[ok] (2) uncached vs cached path agree (max abs err {err:.2e}); gate changes output")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", action="store_true")
    args = ap.parse_args()
    test_math()
    if args.model:
        test_model()
    print("ALL TESTS PASSED")

Writing /kaggle/working/fox_sanity_test.py


In [5]:
!python {CODE_DIR}/fox_patch.py

SmolVLA dir: /usr/local/lib/python3.13/dist-packages/lerobot/policies/smolvla
patched configuration_smolvla.py
patched smolvlm_with_expert.py
patched modeling_smolvla.py


In [6]:
# 2) safety tests (math tests + model-level tests: alpha=0 == baseline, train path == cached path)
!python {CODE_DIR}/fox_sanity_test.py --model

Traceback (most recent call last):
  File "/kaggle/working/fox_sanity_test.py", line 18, in <module>
    from lerobot.policies.smolvla.smolvlm_with_expert import ForgetGate
  File "/usr/local/lib/python3.13/dist-packages/lerobot/policies/__init__.py", line 34, in <module>
    from .groot.configuration_groot import GrootConfig as GrootConfig
  File "/usr/local/lib/python3.13/dist-packages/lerobot/policies/groot/__init__.py", line 19, in <module>
    from .processor_groot import make_groot_pre_post_processors
  File "/usr/local/lib/python3.13/dist-packages/lerobot/policies/groot/processor_groot.py", line 49, in <module>
    from lerobot.datasets.lerobot_dataset import LeRobotDataset
  File "/usr/local/lib/python3.13/dist-packages/lerobot/datasets/__init__.py", line 23, in <module>
    from .aggregate import aggregate_datasets
  File "/usr/local/lib/python3.13/dist-packages/lerobot/datasets/aggregate.py", line 32, in <module>
    from .dataset_metadata import LeRobotDatasetMetadata
  File

In [9]:
# check what's installed
!pip show av | head -3
!python -c "import av; print('av', av.__version__); import av.option; print('av.option OK')"

!pip install -q -U "av>=15,<16"
!python -c "import av, av.option; print('av', av.__version__, 'OK')"

Name: av
Version: 19.0.2
Summary: Pythonic bindings for FFmpeg's libraries.
ERROR: Pipe to stdout was broken
av 19.0.2
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import av; print('av', av.__version__); import av.option; print('av.option OK')
                                            ^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'av.option'
av 15.1.0 OK


In [ ]:
import os, shutil, subprocess, sys

os.environ["WANDB_MODE"] = "disabled"

TRAIN  = shutil.which("lerobot-train")
STEPS  = 20000
OUT    = "/kaggle/working/outputs/smolvla_fox"
RENAME = ('{"observation.images.cam_head":"observation.images.camera1",'
          '"observation.images.cam_wrist":"observation.images.camera2",'
          '"observation.images.cam_side":"observation.images.camera3"}')

cmd = [
    "accelerate", "launch", "--multi_gpu", "--num_processes=2", "--mixed_precision=no", TRAIN,
    "--policy.path=lerobot/smolvla_base",
    "--policy.use_fox=true",
    "--policy.fox_identity_init=true",
    "--policy.fox_bias_init=4.0",
    "--policy.fox_alpha_lr=2e-3",
    f"--policy.scheduler_decay_steps={STEPS}",
    "--policy.push_to_hub=false",
    f"--dataset.repo_id={DATASET}",
    "--dataset.revision=main",
    "--dataset.video_backend=pyav",
    f"--rename_map={RENAME}",
    "--batch_size=8",
    "--num_workers=4",
    f"--steps={STEPS}",
    "--save_freq=2500",
    "--log_freq=50",
    f"--output_dir={OUT}",
    "--job_name=smolvla_fox",
    "--seed=1000",
]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, env=dict(os.environ, PYTHONUNBUFFERED="1"))
for line in proc.stdout:
    print(line, end="")
sys.exit(proc.wait())

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
INFO 2026-11-08 07:17:48 ot_train.py:565 {'accelerator': {'activation_checkpointing': {'mode': <ActivationCheckpointingMode.NONE: 'none'>},
                 'compile': {'backend': 'inductor',
                             'enabled': False,
                             'mode': None,
                             'regional': True},
                 'ddp': {'find_unused_parameters': True,
                         'gradient_as_bucket_view': False,
                         'static_graph': False},
                 'fsdp': {'cpu_offload': False,
                          'ignored_modules': None,
                          'min_num_params': None,
                          'mixed_precision_policy': {'cast_forward_i